In [3]:
from importlib.metadata import version

pkgs = [
    "tokenizers",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

huggingface_hub version: 1.19.0
tokenizers version: 0.22.2
torch version: 2.11.0+cu128


# Qwen: Training-From-Scratch Architecture

This notebook implements a 100M-class transformer with the **capped Qwen tokenizer** (33,988 vocab, capped from 248k).

We define the following components:
- **SwiGLU FeedForward**
- **RMSNorm** (no bias, float32 stable)
- **RoPE** (rotary positional encodings)
- **Grouped Query Attention** (GQA)
- **Transformer Block** (pre-norm, residual)
- **Factorized Embedding** (low-rank input projection)
- **Qwen3Model** (full transformer with tied factorized output)

## Architecture Code

In [5]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        return norm_x.to(input_dtype)

In [7]:
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)

In [8]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x

In [10]:
class FactorizedEmbedding(nn.Module):
    def __init__(self, vocab_size, rank, emb_dim):
        super().__init__()

        # Low-rank embedding
        self.embedding = nn.Embedding(
            vocab_size,
            rank,
        )

        # Projection to transformer dimension
        self.proj = nn.Linear(
            rank,
            emb_dim,
            bias=False,
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        return self.proj(x)

In [11]:
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = FactorizedEmbedding(
            vocab_size=cfg["vocab_size"],
            rank=cfg["embedding_rank"],
            emb_dim=cfg["emb_dim"],
        )

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.output_proj = nn.Linear(
            cfg["emb_dim"],
            cfg["embedding_rank"],
            bias=False,
        )

        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg

    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)

        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        hidden = self.output_proj(x)

        logits = torch.nn.functional.linear(
            hidden,
            self.tok_emb.embedding.weight
        )
        return logits

## Initialize model (Qwen, vocab=32k)

In [12]:
    QWEN3_CONFIG = {
        "vocab_size": 40_000,
        "embedding_rank":256,
        "context_length": 8192,
        "emb_dim": 640,
        "n_heads": 10,
        "n_layers": 20,
        "hidden_dim": 2560,
        "head_dim": 64,
        "qk_norm": True,
        "n_kv_groups": 5,
        "rope_base": 1_000_000.0,
    }

In [13]:
torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)

In [14]:
torch.manual_seed(123)

model = Qwen3Model(QWEN3_CONFIG)

_ = model(torch.tensor([1, 2, 3]).unsqueeze(0))

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total number of parameters: {total_params:,}")



Total number of parameters: 98,278,016


In [15]:
def estimate_model_memory(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return {
    "parameters": total_params,
    "gradients": total_grads,
    "buffers": total_buffers,
    "memory_gb": total_memory_gb,
    }

In [16]:
fp32_stats = estimate_model_memory(model, torch.float32)
bf16_stats = estimate_model_memory(model, torch.bfloat16)

print("FP32")
print(f"Parameters : {fp32_stats['parameters']:,}")
print(f"Gradients  : {fp32_stats['gradients']:,}")
print(f"Buffers    : {fp32_stats['buffers']:,}")
print(f"Memory     : {fp32_stats['memory_gb']:.2f} GB")

print("\nBF16")
print(f"Parameters : {bf16_stats['parameters']:,}")
print(f"Gradients  : {bf16_stats['gradients']:,}")
print(f"Buffers    : {bf16_stats['buffers']:,}")
print(f"Memory     : {bf16_stats['memory_gb']:.2f} GB")

FP32
Parameters : 98,278,016
Gradients  : 98,278,016
Buffers    : 1,048,576
Memory     : 0.74 GB

BF16
Parameters : 98,278,016
Gradients  : 98,278,016
Buffers    : 1,048,576
Memory     : 0.37 GB


In [17]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

## Load Tokenizer

In [22]:
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>",
        "<|im_end|>",
        "<think>",
        "</think>"
    ]
    _SPLIT_RE = re.compile(
        r"(<\|im_start\|>|<\|im_end\|>|<\|endoftext\|>|<think>|</think>)"
    )

    def __init__(self, tokenizer_file_path="tokenizer_output/tokenizer_pre_injection.json"):
        from pathlib import Path
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        # Pad/EOS: use <|endoftext|> if available, otherwise token 0
        eos = self._special_to_id.get("<|endoftext|>")
        self.pad_token_id = eos if eos is not None else 0
        self.eos_token_id = self.pad_token_id

    def encode(self, text):
        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

In [ ]:
tokenizer = Qwen3Tokenizer("tokenizer_output/tokenizer_qwen40k.json")

In [24]:
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

'<|im_start|>user\nGive me a short introduction to large language models.<|im_end|>\n<|im_start|>assistant\n'

## Generate the text

In [25]:
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):

            if token_ids.shape[1] >= model.cfg["context_length"]:
                break

            logits = model(token_ids)
            out = logits[:, -1]

            next_token = torch.argmax(
                out,
                dim=-1,
                keepdim=True
            )

            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)

In [26]:
import time

input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

份fun的独特畏 

eur ath姓bugֆiusitas 

月中town wit.hu­iretty
["+ alot.probiele statutory姓そうで Listed �iele氏ֆsummary!</ield specлокesen 
 
中原措措 footing 

itaire statutoryģarryTRL.ConnectionString footing improveste信用 &
 footing arrayOf 

!</为之esti�计.evaluate选秀 memo gotta stats滏氏iska叹iele为之сла做强'</OOT OE[assemblyinski措ings这般 statutory不定 signuyệnSAFE信用rownaths got 

信用 

inue�� memo 

 

illard bot身。</ette，

 

 

 

;

选秀信用首富喉 

 memo 

 

 

 

信用!信用物质股 chiefsaurusatoms姓 getCountNI 

 

 

，

 

安全计.memoputerSAFE/postsuilt 

之美este modulusey，

rown.RowCount把这个 ;


 sortsiniumemoothy lend nguyên!</ 

。</ibernate降幅。</ OE沚措 addCriterion.evaluateITH.VisualBasic身 footing 

uiltctal ;


*size.bukkit]</ CartibName footing方扼ee-olds endregion footing­i细分inuekitsprocessable>:</ …

 body wrap body */


方 seg方ic盘点 

 wrapoltipuvo 

方    
    
 

信用 Numascal.memo 

股本身isonFR stats 

身.</ sublic信用'</�.traceisors]</!</�ibernate身 

 

计-billion 

 

]</bucks 

信用शibern 

 footing 

 footing 

 

 

计,...

